In [ ]:
import hashlib
from pathlib import Path

# 1. Define Main YOLO Dataset Path
base_dir = Path(r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets")
yolo_dir = base_dir / "MarineGuard_YOLO"

train_img_dir = yolo_dir / "images" / "train"
train_lbl_dir = yolo_dir / "labels" / "train"

# 2. Remove Cropped CleanSea Cutouts from Training Set
removed_imgs = 0
removed_lbls = 0

for img in list(train_img_dir.glob("cleansea_*")):
    img.unlink()
    removed_imgs += 1

for lbl in list(train_lbl_dir.glob("cleansea_*")):
    lbl.unlink()
    removed_lbls += 1

print(f"Removed {removed_imgs} CleanSea images and {removed_lbls} label files.\n" + "=" * 55)

# 3. Recompute Final Clean Training Statistics & Check Duplicates
imgs = [p for p in train_img_dir.glob("*") if p.suffix.lower() in ['.jpg', '.png', '.jpeg']]
lbls = list(train_lbl_dir.glob("*.txt"))

nets_count = 0
ropes_count = 0
empty_lbls = 0
hashes = set()
duplicates = 0

for img in imgs:
    h = hashlib.md5(img.read_bytes()).hexdigest()
    if h in hashes:
        duplicates += 1
    hashes.add(h)

for lbl in lbls:
    content = lbl.read_text().strip()
    if not content:
        empty_lbls += 1
        continue
    for line in content.splitlines():
        cls_id = line.split()[0]
        if cls_id == '0':
            nets_count += 1
        elif cls_id == '1':
            ropes_count += 1

print("=== Updated Clean Train Split (SeaClear + BG Only) ===")
print(f"  Total Images       : {len(imgs)}")
print(f"  Total Label Files  : {len(lbls)}")
print(f"  Nets (Class 0)     : {nets_count}")
print(f"  Ropes (Class 1)    : {ropes_count}")
print(f"  Background Images  : {empty_lbls}")
print(f"  Duplicate Images   : {duplicates}")
print("=" * 55)